# Routing

While a sequential workflow, as seen in the Prompt Chaining pattern, is foundational to think about multi-step agent applications, it has several limitations:
- It lacks the ability to make decisions based on context. 
- Without a mechanism to choose the correct tool or sub-process for a specific task, the system remains rigid and non-adaptive. 
- It makes difficult to build sophisticated applications that can manage the complexity and variability of real-world user requests.

The Routing pattern provides a solution by introducing conditional logic, enabling the system to first analyze an incoming query to determine its intent or nature. Based on this analysis, the agent dynamically directs the flow of control to the most appropriate specialized tool, function, or sub-agent.



## Implementation with Flyte V2

This notebook refactors the original LangChain + RunnableBranch routing example into a Flyte v2 task that uses LangGraph for the routing logic. We keep the same three logical handlers (`booking_handler`, `info_handler`, `unclear_handler`), but:

- Use **LangGraph** to implement the router and delegation graph.
- Use **Flyte v2** to wrap the routing logic in async tasks (`route_request`, `route_requests_batch`).
- Make it easy to run both locally and remotely via Flyte.

1. Install dependencies

In [ ]:
!uv pip install flyte[tui] langgraph openai

2. Export your API key 

In [ ]:
!export OPENAI_API_KEY=<your-api-key>

3. Import dependencies and declare the resources your execution environment will need, using the `TaskEnvironment` class. This allows you to allocate precise resources and run agents on their own container with all dependencies baked in automatically:

In [ ]:
from typing import TypedDict, Literal

import asyncio
import flyte
from flyte import TaskEnvironment, Resources, Secret
from langgraph.graph import StateGraph, END
from openai import AsyncOpenAI


# --- Flyte TaskEnvironment configuration ---

env = TaskEnvironment(
    name="routing_env",
    resources=Resources(cpu="1", memory="1Gi"),
    cache="auto",
    # Flyte will build the container image with the dependencies using smart layering.
    image=flyte.Image.from_debian_base().with_pip_packages("openai", "langgraph")
)

# Single OpenAI async client shared within the container.
client = AsyncOpenAI()  # reads OPENAI_API_KEY from the environment

5. Define the stateful Langgraph router we'll use to track the request, analysis and routing decision. The decision is made by prompting an LLM. Other decision-making methods use a rule-based approach or embedded-based semantic similarity.

In [ ]:


# --- LangGraph state and graph definition ---

class RouterState(TypedDict, total=False):
    request: str
    decision: Literal["booker", "info", "unclear"]
    output: str

async def router_node(state: RouterState) -> RouterState:
    """Call the LLM to decide how to route the request."""
    request = state["request"]
    messages = [
        {
            "role": "system",
            "content": (
                "Analyze the user's request and determine which specialist handler should process it.\n"
                "- If the request is related to booking flights or hotels, output 'booker'.\n"
                "- For general information questions, output 'info'.\n"
                "- If the request is unclear or doesn't fit either category, output 'unclear'.\n"
                "ONLY output one word: 'booker', 'info', or 'unclear'."
            ),
        },
        {"role": "user", "content": request},
    ]
    resp = await client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=messages,
    )
    decision = (resp.choices[0].message.content or "").strip().lower()
    if decision not in ("booker", "info", "unclear"):
        decision = "unclear"
    # Preserve existing fields and add decision
    return {**state, "decision": decision}


5. Define the subagents that will handle different types of requests:

In [ ]:
def booking_handler(request: str) -> str:
    """Simulates the Booking Agent handling a request."""
    return f"Booking Handler processed request: '{request}'. Result: Simulated booking action."


def info_handler(request: str) -> str:
    """Simulates the Info Agent handling a request."""
    return f"Info Handler processed request: '{request}'. Result: Simulated information retrieval."


def unclear_handler(request: str) -> str:
    """Handles requests that couldn't be delegated."""
    return f"Coordinator could not delegate request: '{request}'. Please clarify."


6. Now we wire our handlers into a LangGraph `StateGraph` state machine:
- Each node (`booking_node`, `info_node`, `unclear_node`) reads the `request` from `RouterState` and writes an `output` using the matching handler.
- The `router_node` (above) sets a `decision` of `"booker"`, `"info"`, or `"unclear"`, and `route_decision` simply returns that field.
- We add all nodes to the graph, set `"router"` as the entrypoint, add conditional edges from `"router"` to the three nodes based on `decision`, and connect each of them to `END`.

Compiling with `graph.compile()` gives us a single callable app that Flyte tasks (or plain Python) can use to route any request.

In [ ]:
async def booking_node(state: RouterState) -> RouterState:
    return {**state, "output": booking_handler(state["request"])}


async def info_node(state: RouterState) -> RouterState:
    return {**state, "output": info_handler(state["request"])}


async def unclear_node(state: RouterState) -> RouterState:
    return {**state, "output": unclear_handler(state["request"])}


def route_decision(state: RouterState) -> Literal["booker", "info", "unclear"]:
    return state["decision"]


graph = StateGraph(RouterState)

# Nodes
graph.add_node("router", router_node)
graph.add_node("booker", booking_node)
graph.add_node("info", info_node)
graph.add_node("unclear", unclear_node)

# Entry point and conditional edges
graph.set_entry_point("router")
graph.add_conditional_edges(
    "router",
    route_decision,
    {
        "booker": "booker",
        "info": "info",
        "unclear": "unclear",
    },
)

# Terminal edges
graph.add_edge("booker", END)
graph.add_edge("info", END)
graph.add_edge("unclear", END)

# Compile the LangGraph app
app = graph.compile()


6. Wrap the Langgraph app in a Flyte task enabling it to run with predefined resources, dependencies and caching behavior:

In [ ]:
@env.task
async def route_request(request: str) -> str:
    """Route a single request through the LangGraph app and return the handler output."""
    final_state = await app.ainvoke({"request": request})
    return final_state["output"]

7. Run the task:

In [ ]:
if __name__ == "__main__":
    if "OPENAI_API_KEY" not in os.environ:
        print("Set OPENAI_API_KEY in your environment before running this demo.")
    else:
        run = flyte.with_runcontext(mode="local").run(route_request("What is the capital of France?"))
        print(run.outputs)


### Remote execution

[**Optional**] Configure your connection to a remote Flyte cluster. This tells Flyte where to run your workflows and how to build container images.

**Configuration Options:**
- `endpoint`: Your Flyte cluster URL
- `org`: Your organization name
- `project`: Project to organize workflows
- `domain`: Environment (development, staging, production)
- `image_builder`: Use "remote" to build images on the cluster (no local Docker required)

In [ ]:
import flyte
flyte.init(
    endpoint="<your-union-endpoint-url>",  # Your Union cluster URL
    org="<your-union-org>",                                     # Your organization
    project="<your-project>",                               # Your project name
    domain="development",                           # Environment: development/staging/production
    image_builder="remote",                         # Build images on cluster (no local Docker needed)
    auth_type="DeviceFlow",
)

if __name__ == "__main__":
    flyte.init_from_config()
    if "OPENAI_API_KEY" not in os.environ:
        print("Set OPENAI_API_KEY in your environment before running this demo.")
    else:
        run = flyte.run(route_request("What is the capital of France?"))
        print(run.outputs)

## Scaling the pattern

1. Make it batchable for multiple requests:

In [ ]:
@env.task
async def route_requests_batch(requests: list[str]) -> list[str]:
    """Route a batch of requests in parallel with bounded concurrency."""
    sem = asyncio.Semaphore(20)

    async def _one(req: str) -> str:
        async with sem:
            return await route_request(req)

    tasks = [_one(r) for r in requests]
    return list(await asyncio.gather(*tasks))